# Did the August-2026 campaign actually run as designed?

This notebook checks the experiment folder `climate devices experiments data/` against
the written protocol in **`REGLAS_EXPERIMENTO.pdf`**, and plots what was recorded.

All the logic lives in **`protocol_check.py`** beside this notebook — the notebook is the
narrative, the module is the implementation. Re-run top to bottom; it takes about a minute.

## What has to be reconstructed first

The folder holds one CSV per InfluxDB series in **event-logged** form: a row appears only
when a value *changes*. Nothing in it says which protocol block a row belongs to. So before
anything can be compared with the prescription, three things have to be decoded:

| # | Fact | Consequence if you miss it |
|---|---|---|
| 1 | `_time` is **UTC**; the protocol is written in **Murcia local time** (CEST = UTC+2) | Every block looks 2 h off the 00:00/03:00/…/21:00 grid and nothing matches |
| 2 | `Fog_0.csv` encodes a **duty cycle** — `_value == 1` means *spraying*; integrating it per minute recovers the protocol's own 5s/10s/20s axis. `-15` is the idle code | FOG looks like a binary device with no levels |
| 3 | `Ventana*` / `Pantalla*` report **position**, so they ramp 0 → setpoint. The commanded level is the *sustained* value, not the instantaneous reading. `-1`, `-3` are fault codes | Every window block reads as a smear of values from 0 to its setpoint |

Fact 1 is the one that silently breaks everything, so it is worth seeing for yourself — the
next-to-last cell of section 1 demonstrates it.

## Files ignored, and why

| File | Reason |
|---|---|
| `Calefactores_1.csv`, `Recirculador_1.csv` | 2-row duplicate export fragments |
| `Pantalla_de_sombreo_lateral_sur_0.csv` | 18 rows over 13 s; not part of the protocol |
| `Calefactores_0` + `Calefactores_2`, `Recirculador_0` + `Recirculador_centro_0` | duplicate exports of one device each — fused (verified below to agree 100 %) |


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import protocol_check as pk

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 90)
plt.rcParams.update({"figure.dpi": 110, "savefig.bbox": "tight", "font.size": 9})

print("campaign day 1 =", pk.DAY1.date(), " | UTC offset applied:", pk.UTC_OFFSET)
for ph, (a, b) in pk.PHASES.items():
    print(f"  Phase {ph}: days {a}-{b}  = "
          f"{(pk.DAY1 + pd.Timedelta(days=a-1)).date()} to {(pk.DAY1 + pd.Timedelta(days=b-1)).date()}")

## 1. Load and decode

`run_all()` does every step: load, decode, recover per-block levels, check all four phases,
fit the step responses, scan the safety rules.

In [ ]:
rep = pk.run_all()

### The duplicate-export check

The two `Calefactores` files and the two `Recirculador` files are claimed to be exports of the
same device. Worth proving rather than assuming, because fusing two genuinely different
devices would corrupt RECIRC and RECIRC2 everywhere.

In [ ]:
for group, a, b in [("RECIRC", "Recirculador_0.csv", "Recirculador_centro_0.csv"),
                    ("RECIRC2", "Calefactores_0.csv", "Calefactores_2.csv")]:
    sa = pk._read_events(pk.DATA_DIR / a).resample("5min").ffill()
    sb = pk._read_events(pk.DATA_DIR / b).clip(lower=0).resample("5min").ffill()
    j = pd.concat([sa.rename("a"), sb.rename("b")], axis=1).ffill().dropna()
    print(f"{group:8s} {a} vs {b}: {len(j)} 5-min bins, agree {100*(j.a==j.b).mean():.2f} %")

### Fact 1, demonstrated: the timestamps are UTC

FOG is the cleanest witness, because its duty cycle is unmistakable. Below are the hours in
which the fog sprayed, first as stored and then shifted by +2 h. Only one of the two lands on
the protocol's 3-hour grid (00:00, 03:00, 06:00 … 21:00).

In [ ]:
raw = pd.read_csv(pk.DATA_DIR / "Fog_0.csv")
t_utc = pd.to_datetime(raw["_time"], format="mixed", utc=True).dt.tz_localize(None)
spray = pd.Series((pd.to_numeric(raw["_value"], errors="coerce") == 1).astype(float).values,
                  index=t_utc.values).sort_index()
spray = spray[~spray.index.duplicated(keep="last")]
duty_utc = spray.resample("1s").ffill().resample("1min").sum().resample("1h").mean()

day = duty_utc.loc["2026-08-12"]
active = day[day > 1]
print("hours with fog, as stored (UTC)      :", sorted(active.index.hour.tolist()))
print("hours with fog, shifted +2 h (local) :", sorted(((active.index.hour + 2) % 24).tolist()))
print("protocol block starts                : [0, 3, 6, 9, 12, 15, 18, 21]")

The shifted version is exactly the block grid; the stored version is not. Everything from
here on is in local time.

### Fact 2, demonstrated: FOG's levels are a duty cycle

Seconds sprayed per minute, over every minute the fog was on. Three clusters, exactly where
the protocol's `5s / 10s / 20s` says they should be.

In [ ]:
per_min = rep.fog[rep.fog > 0]
fig, ax = plt.subplots(figsize=(7.5, 3))
ax.hist(per_min.values, bins=np.arange(0.5, 22.5, 1), color="#2E86AB", alpha=0.85)
for lv, lab in [(5, "5 s"), (10, "10 s"), (20, "20 s")]:
    ax.axvline(lv, color="#C1443E", ls="--", lw=1)
    ax.text(lv, ax.get_ylim()[1] * 0.95, lab, ha="center", fontsize=8, color="#C1443E")
ax.set_xlabel("seconds spraying per minute")
ax.set_ylabel("minutes")
ax.set_title("FOG intensity is a duty cycle, and it takes the three prescribed values",
             fontsize=10, loc="left")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.show()

### Fact 3, demonstrated: the windows ramp to their setpoint

One Phase-D block commanded to `VENT_ROOF 25`. The raw series travels 0 → 25 over the first
minutes; only the plateau is the commanded level. Reading the mean, or any single sample from
the first minutes, would give the wrong answer.

In [ ]:
t0 = pd.Timestamp("2026-08-21 03:00")
cols = [c for c in rep.act.columns if c.startswith("VENT_ROOF::")]
seg = rep.act.loc[t0 - pd.Timedelta(minutes=10):t0 + pd.Timedelta(minutes=50), cols]

fig, ax = plt.subplots(figsize=(8, 3))
for c in cols:
    ax.step(seg.index, seg[c], where="post", lw=1.4, label=c.split("::")[1])
ax.axvline(t0, color="#999999", ls="--", lw=1)
ax.axvline(t0 + pk.RAMP_SKIP, color="#1B7F5F", ls=":", lw=1.2)
ax.text(t0 + pk.RAMP_SKIP, 12, "  level read after here", fontsize=7.5, color="#1B7F5F")
ax.set_ylabel("aperture (%)")
ax.set_title(f"VENT_ROOF ramping to its setpoint  ·  block {t0:%d %b %H:%M}  ·  "
             f"recovered level = {rep.blocks.loc[t0, 'VENT_ROOF']:g} %", fontsize=10, loc="left")
ax.legend(fontsize=7, frameon=False)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.show()

print("raw samples in the treatment window:",
      sorted(seg.loc[t0:t0 + pd.Timedelta(hours=2), cols[0]].dropna().unique().tolist())[:12], "...")

## 2. Data quality — read this before trusting any result

Two checks matter here. Range checks catch impossible values. But the failure that actually
happened in this campaign is the one a range check cannot see: **a channel that keeps
reporting a perfectly plausible constant for ever.**

In [ ]:
display(rep.quality)

### Six sensors froze within six minutes of each other

`temp_alta`, `temp_baja`, `hum_alta`, `hum_baja`, `par` and `rad` all stop changing on
**2026-08-22, between 09:04 and 09:10** and never move again. Six sensors failing inside six
minutes is one acquisition/bus failure, not six sensor faults. `temp_centro`, `hum_centro`,
`co2` and `dpv` survive — presumably a different channel.

This matters because the frozen values sit **inside** the plausible range (27.9 °C, 80 % RH,
542 µmol PAR), so nothing but this check flags them. PAR frozen at 542 means *permanent
daylight* for the rest of the record. `load_sensors()` masks them from the freeze onward, so
they show as missing rather than silently contributing a constant to every later average.

In [ ]:
display(rep.flatlines)

t_dead = rep.flatlines.loc[rep.flatlines.dead, "frozen_from"].min()
print("first freeze:", t_dead, "= campaign day", pk.campaign_day(t_dead))
print("Phase D spans days", pk.PHASES["D"], "- so most of the largest phase has centre sensors only.")

raw_alta = pk._read_events(pk.DATA_DIR / pk.SENSORS["temp_alta"]).resample("10min").mean()
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(raw_alta.index, raw_alta.values, lw=0.6, color="#C1443E", label="temp_alta (raw)")
ax.plot(rep.sens.index, rep.sens["temp_centro"], lw=0.6, color="#1B7F5F",
        alpha=0.8, label="temp_centro (alive)")
ax.axvline(t_dead, color="#000000", lw=1.2)
ax.annotate("channel freezes at 27.9 °C\nand never moves again",
            xy=(t_dead, 27.9), xytext=(20, 45), textcoords="offset points", fontsize=8,
            arrowprops=dict(arrowstyle="->", lw=0.8))
ax.set_ylabel("°C")
ax.set_title("The failure a range check cannot see", fontsize=10, loc="left")
ax.legend(fontsize=7.5, frameon=False, loc="lower left")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.show()

### Actuator status codes

Negative values in the actuator files are status/fault codes, not levels, and are masked.
The window `-1` / `-3` codes are worth a glance: they mark position-unknown stretches, some
lasting 15 minutes inside a treatment block.

In [ ]:
display(rep.status_codes)
print("Ignored on purpose:")
for f, why in pk.IGNORED_FILES.items():
    print(f"  {f:42s} {why}")

## 3. Per-block levels: the table everything else is built on

One row per 3-hour block, with the commanded level recovered for each of the 7 protocol
groups, plus the level during the 3rd (washout) hour and a per-group device spread
(`0` = all devices in the group agree).

In [ ]:
names = list(pk.GROUPS)
view = rep.blocks[["day", "phase"] + names + ["n_active", "washout_clean"]]
display(view.head(24))
print("blocks recovered:", len(rep.blocks))
print("off-grid readings (not within tolerance of any legal level):",
      int(rep.blocks[[f'{n}_offgrid' for n in names]].sum().sum()))
print("blocks where devices inside a group disagreed:",
      int((rep.blocks[[f'{n}_spread' for n in names]].fillna(0) > 6).any(axis=1).sum()))

## 4. The campaign at a glance

Seven actuator lanes under the interior climate, with the phase boundaries and the sensor
failure marked. This single figure is the fastest way to see that the programme ran, where it
stopped, and what the greenhouse did in response.

In [ ]:
fig = pk.plot_campaign_overview(rep)
plt.show()

## 5. Coverage

Every expected block is present in all four phases — there is no missing logging. What the
`blocks_all_off` column shows instead is idle time *inside* the phases.

In [ ]:
display(rep.coverage)

idle = rep.blocks[(rep.blocks.phase == "D") & (rep.blocks[names].fillna(0).sum(axis=1) == 0)]
print(f"Phase D has {len(idle)} all-off blocks out of {len(rep.blocks[rep.blocks.phase=='D'])}.")
print("The last", len(idle[idle.index >= '2026-09-01']), "of them are the final two days:")
print(sorted(set(idle[idle.index >= '2026-09-01'].index.date)))

**The campaign stops early.** Days 30–31 (1–2 September) are 16 consecutive all-off blocks —
Phase D was scheduled to run through day 31 but the programme actually ends on 31 August
(day 29). Sporadic multi-group activity on 3–6 September sits outside the campaign window and
looks like manual testing, not Phase D.

## 6. Phase A — characterisation (days 1–5)

Phase A does **not** use the 3-hour block grid: each trial holds one group on until steady
state (~4 h). So the trials are located by detecting the steps themselves and only then
matched against the prescription, one observed step per prescribed trial.

In [ ]:
display(rep.phase_a)
print(rep.phase_a.verdict.value_counts().to_string())

Four findings here:

- **A1 fails.** Day 1 was supposed to be 24 h of passive baseline. Instead all five windows
  cycled between 0 and 100 % for 92 minutes between 11:00 and 14:49, roof and side moving
  together — the house's own overtemperature response, not the manual programme. There is
  also no actuator logging at all before 11:00 on day 1.
- **A2 ran 4 h late** (10:04 instead of 06:00) — into the morning heat the revised protocol
  moved it to 06:00 specifically to avoid.
- **A6 (SHADE) ran 6 h late**, at 14:04 instead of 08:00. The protocol allows a midday
  fallback, but only *"con las ventanas abiertas como estado de referencia"*. Check what the
  reference state actually was:
- **A8 never ran.** The only fog step on day 4 is A7's at 06:00; there is no 18:00 step.
  A9 (RECIRC) then overlaps A7's fog, so it is not the isolated single-group trial Phase A
  requires — an overlap the written schedule itself creates on days 3 and 4.

In [ ]:
t = pd.Timestamp("2026-08-05 14:04")
print("A6 SHADE fallback at 14:04 — reference state during the trial:")
for g in names:
    lv = pk.group_level(rep.act, rep.fog, pk.GROUPS[g], t, t + pd.Timedelta(hours=2))
    print(f"  {g:10s} {lv['level']}")
print("\nThe protocol's midday fallback requires the windows OPEN as the reference state.")
print("VENT_ROOF and VENT_SIDE were both at 0, so the fallback ran without its own precondition.")

print("\nOverlapping trials in the written schedule (violates 'un solo grupo cada vez'):")
for a, b in [("A4", "A6"), ("A7", "A9")]:
    ra = next(x for x in pk.PHASE_A if x["id"] == a)
    rb = next(x for x in pk.PHASE_A if x["id"] == b)
    print(f"  {a} day {ra['day']} {ra['hour']:02d}:00 held ~4 h  overlaps  "
          f"{b} day {rb['day']} {rb['hour']:02d}:00")

### The number Phase A exists to produce: tau

Protocol section 3 says the 2 h + 1 h block duration is *provisional* until tau is measured
here. Measuring it needs care: in an **empty greenhouse in August the diurnal swing (~20 °C
per day) is larger than any actuator effect**, so fitting a bare step

$$y = y_0 + K\,(1 - e^{-t/\tau})$$

to a 4-hour window starting at 06:00 mostly fits sunrise. The fit then drives tau and K to
their bounds and the resulting "tau" is an artefact. `fit_tau` therefore fits a background
trend alongside the step,

$$y = y_0 + c\,t + K\,(1 - e^{-t/\tau})$$

and rejects any fit that hits its bounds, returns a tau longer than half the observation
window (not identifiable from that window), or implies an implausible gain for a single
actuator.

In [ ]:
display(rep.dynamics)
usable = rep.dynamics[rep.dynamics.usable & (rep.dynamics.r2 > 0.8)]
print(f"{len(usable)} of {len(rep.dynamics)} fits are usable.\n")
import json
print(json.dumps(pk.tau_summary(rep.dynamics), indent=2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
for ax, detrend, title in [(axes[0], False, "bare step fit — fits sunrise"),
                           (axes[1], True, "with a trend term — fits the step")]:
    onset = pd.Timestamp("2026-08-05 06:04")     # A4, at dawn
    d = pk.fit_tau(rep.sens, onset, "temp_centro", detrend=detrend)
    seg = rep.sens.loc[onset:onset + pd.Timedelta(hours=4), "temp_centro"].dropna()
    tt = (seg.index - seg.index[0]).total_seconds().values / 60
    ax.plot(seg.index, seg.values, lw=1.2, color="#C1443E", label="measured")
    fit = d["baseline"] + d["gain"] * (1 - np.exp(-tt / d["tau_min"]))
    if detrend:
        fit = fit + d["trend_per_h"] / 60 * tt
    ax.plot(seg.index, fit, lw=1.6, ls="--", color="#1B7F5F", label="fit")
    ax.set_title(f"A4 · {title}\nτ={d['tau_min']:.0f} min, gain={d['gain']:+.1f} °C, "
                 f"R²={d['r2']:.2f}, usable={d['usable']}", fontsize=8.5, loc="left")
    ax.legend(fontsize=7, frameon=False)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
plt.show()
print("Both fits have a high R^2. R^2 mostly measures the trend here, which is exactly why it")
print("cannot be the acceptance criterion - and why A4 is rejected in the table above.")

In [ ]:
fig = pk.plot_step_fit(rep, "2026-08-05 18:04")   # A5, evening: clean step, low solar forcing
plt.show()

**Answer to the protocol's open question.** From the usable temperature fits, tau is roughly
**2–39 min (median ≈ 16 min)**. Against the longest, tau ≈ 39 min:

- the **2 h treatment is ≈ 3.1 tau — adequate** (the treatment does reach steady state);
- the **1 h washout is only ≈ 1.5 tau — too short.** The protocol itself asks for 3–5 tau to
  return to baseline. At 1.5 tau about 22 % of the previous treatment is still present when
  the next block begins, so consecutive blocks carry over into one another.

That is the one change to the design this data actually justifies: keep the 2 h treatment,
lengthen the washout to at least 2 h.

## 7. Phase B — dose–response (days 6–12)

Five prescribed level sequences, run back to back, one group at a time. The protocol does not
say which block the phase starts on, so the concatenated expected sequence is slid over the
observed blocks and the best alignment is taken.

In [ ]:
print(rep.phase_b_summary)
print()
print(rep.phase_b.groupby("group", sort=False).agg(
    matched=("match", "sum"), blocks=("match", "size")).to_string())
print("\nMismatched blocks:")
display(rep.phase_b[~rep.phase_b.match])

**58 of 60 blocks exact**, every block single-group as required, every washout clean. FOG,
LIGHTS and SHADE are perfect (12/12, 15/15, 9/9).

The two misses are both a `50 %` command that never executed — the windows sat at exactly 0
for the whole block, with the neighbouring blocks working normally. So these are two skipped
controller rows, not a data gap and not a stuck actuator. Neither block was hot enough for the
safety rule to explain it (`safety_hot` is False for both).

In [ ]:
for t0, grp in [(pd.Timestamp("2026-08-08 06:00"), "VENT_ROOF"),
                (pd.Timestamp("2026-08-10 21:00"), "VENT_SIDE")]:
    cols = [c for c in rep.act.columns if c.startswith(grp + "::")]
    seg = rep.act.loc[t0 - pd.Timedelta(hours=1):t0 + pd.Timedelta(hours=4), cols]
    fig, ax = plt.subplots(figsize=(9, 2.6))
    for c in cols:
        ax.step(seg.index, seg[c], where="post", lw=1.3, label=c.split("::")[1])
    ax.axvspan(t0, t0 + pd.Timedelta(hours=2), color="#C1443E", alpha=0.10)
    ax.text(t0, 60, "  block commanded 50 %", fontsize=8, color="#C1443E")
    ax.set_ylim(-5, 105)
    ax.set_ylabel("aperture (%)")
    ax.set_title(f"{grp} skipped its 50 % command · block {t0:%d %b %H:%M}", fontsize=10, loc="left")
    ax.legend(fontsize=7, frameon=False, ncol=3)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    plt.show()

In [ ]:
fig = pk.plot_dose_response(rep, "VENT_ROOF")
plt.show()
fig = pk.plot_dose_response(rep, "FOG")
plt.show()

## 8. Phase C — fractional factorial (days 13–18)

32 runs of a resolution-IV design, all 7 groups set per run. Each observed block's 7-level
vector is matched against the design table, in the prescribed randomised order.

In [ ]:
print(rep.phase_c_summary)
display(rep.phase_c[~rep.phase_c.exact])

**31 of 32 runs exact**, 98.7 % of individual cells.

The single mismatch is not a failure — it is the safety system working. Run 30 (19 Aug 15:00)
was prescribed `VENT_ROOF 0, VENT_SIDE 0, SHADE 100` and instead ran
`VENT_ROOF 100, VENT_SIDE 100, SHADE 0`, which is **exactly** the overtemperature reaction in
protocol section 8. `peak_temp` for that block confirms it breached the 45 °C limit.

This is also the cleanest proof anywhere in the campaign that the overtemperature rule was
genuinely armed in the controller — the programme said "closed", the house opened anyway.

In [ ]:
row = rep.phase_c[~rep.phase_c.exact].iloc[0]
print(f"run {row.run} · block {row.block} · peak temperature {row.peak_temp} °C · "
      f"breached 45 °C: {row.safety_hot}")
fig = pk.plot_block(rep, row.block)
plt.show()

Phase C also started a day later than its nominal day-13 boundary (16 Aug, not 15 Aug),
because Phase B's 60 blocks overrun into day 13. Four blocks on 15 Aug sit idle between the
two phases. That is arithmetic in the design, not an execution error: 60 Phase-B blocks is
7.5 days but Phase B is allotted 7.

## 9. Phase D — random combinations (days 19–31)

`fase_D_programa.csv` is **not in the repo**, so only the 24 blocks tabulated in the PDF can be
verified. Blocks 25–96 cannot be checked at all without that file.

In [ ]:
print(rep.phase_d_summary)
print("\nmismatches:", len(rep.phase_d[~rep.phase_d.exact]))
display(rep.phase_d.head(24))

**24 of 24 exact, 100 % of cells** — including the 25 % / 75 % aperture levels and the 5 s /
10 s fog levels that only appear in Phase D. Whatever generated the schedule and whatever
executed it agree perfectly over the window that can be checked.

Phase D began on 20 Aug (day 18), a day *before* its nominal boundary, following straight on
from Phase C's 32nd run.

To check the rest, regenerate the schedule with the seeded script named in the protocol and
save it as `fase_D_programa.csv`; then:

```python
prog = pd.read_csv("fase_D_programa.csv")          # columns: LIGHTS, FOG, VENT_ROOF, ...
design = prog[pk._C_COLS].values.tolist()
res, summary = pk._match_design(rep.blocks, design, (19, 31), "Phase D (full 96)")
```

## 10. Safety rules (section 8)

These are the only sensor-triggered rules, and the only ones allowed to override the
programme. Two things are being asked here: how often did the conditions arise, and did the
prescribed reaction follow?

In [ ]:
ot = rep.safety[rep.safety.rule == "overtemp"]
otc = ot[ot.start.map(pk.campaign_day) <= 31]
print(f"Above 45 °C during the campaign: {len(otc)} episodes, "
      f"{int(otc.minutes.sum())} minutes ({otc.minutes.sum()/60:.0f} h), peak {otc.peak.max()} °C")
print()
print(otc.attribution.value_counts().to_string())

### How to read this honestly

The scheduled programme opens the roof vents to 100 % routinely, so **an opening during a hot
spell cannot be credited to the safety rule** in general. The informative cases are the
negatives: if the vents stayed shut through a breach, the rule did not act — whatever was
programmed.

- **34 episodes, 2211 minutes (37 h) above 45 °C with the roof vents never reaching 100 %.**
  That is clean evidence the overtemperature rule did not act.
- In several of those the lights were still at 5, 8 or 10 while the house was above 45 °C.
  The rule says `LIGHTS 0`.
- Two episodes prove the rule *was* armed: day 1 (nothing programmed, vents cycled anyway) and
  Phase C run 30. So the rule exists but fires inconsistently, with delays from 0 to 223 min.

Peak recorded temperature was **54.7 °C** during the campaign and **56.4 °C** on 5 September.

In [ ]:
clean = otc[otc.attribution.str.startswith("clean")]
display(clean[["start", "minutes", "peak", "roof_max", "side_max", "shade_min", "lights_max"]])

lit = clean[clean.lights_max > 0]
print(f"\n{len(lit)} of those episodes had LIGHTS still on above 45 °C "
      f"(rule says LIGHTS -> 0): max level {lit.lights_max.max():g}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.2))
d = ot.assign(day=ot.start.map(pk.campaign_day)).groupby("day").minutes.sum()
colors = ["#C1443E" if k <= 31 else "#BBBBBB" for k in d.index]
ax.bar(d.index, d.values, color=colors)
ax.axvline(31.5, color="#999999", ls="--", lw=1)
ax.text(31.7, d.max() * 0.9, "after the\ncampaign", fontsize=7.5, color="#777777")
ax.set_xlabel("campaign day")
ax.set_ylabel("minutes > 45 °C")
ax.set_title("Time above the 45 °C safety limit, by day", fontsize=10, loc="left")
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.show()

In [ ]:
cd = rep.safety[rep.safety.rule == "condensation"]
print("Condensation rule (RH = 100 % for > 20 min with fog on -> fog off):")
print(cd.reaction.value_counts().to_string())
display(cd[["start", "minutes", "peak", "fog_during_s_per_min", "fog_after_s_per_min", "reaction"]])

9 of 13 condensation episodes ended with the fog going off as prescribed. Three did not —
most severely 24 Aug, where the humidity sat at 100 % for **267 minutes** while the fog kept
spraying at ~9 s/min.

## 11. Verdict

| Phase | Days | Result |
|---|---|---|
| **A** — characterisation | 1–5 | **4 of 11 trials deviate.** A1 baseline contaminated by automatic venting; A2 4 h late; A6 6 h late and without its required open-window reference; A8 never ran; A9 not isolated |
| **B** — dose–response | 6–12 | **58/60 blocks exact.** Two skipped 50 % window commands. All blocks single-group, all washouts clean |
| **C** — factorial | 13–18 | **31/32 runs exact.** The one mismatch is the overtemperature rule correctly overriding run 30 |
| **D** — random fill | 19–31 | **24/24 verifiable blocks exact**, but only 24 of 96 can be checked (`fase_D_programa.csv` missing), and the campaign stops on day 29 |

**The execution of the schedule is excellent** — where the prescription can be checked
block-by-block, agreement is 97–100 %. The problems are elsewhere:

1. **Six sensors froze on 22 August** (day 20) at plausible constant values. Most of Phase D
   has centre sensors only, and any analysis that reads `temp_alta`, `temp_baja`, `hum_alta`,
   `hum_baja`, `par` or `rad` after that date is reading a constant. This is the most damaging
   finding, because it is silent.
2. **The 1 h washout is too short** — about 1.5 tau against the 3–5 tau the protocol asks for,
   so consecutive blocks contaminate each other. The 2 h treatment is fine.
3. **The overtemperature rule fires inconsistently**: 37 h above 45 °C with the vents shut,
   sometimes with the lights still on.
4. **The campaign ended two days early** (last active day 31 August = day 29).
5. **Phase A did not deliver clean single-group step responses**, partly because the written
   schedule overlaps trials on days 3 and 4, partly because A2/A6 slipped into the heat where
   the diurnal ramp swamps the signal.

### What this means for the modelling notebooks

`README.md` already flags that the actuator effects in `03. what-if control.ipynb` and
`04.RL control.ipynb` are not causally identifiable from observational data. This campaign was
the fix — and Phases B, C and D deliver exactly the randomised, program-driven actuator
variation those notebooks need. Two cautions when you use it:

- **Scope to 8–31 August and prefer the centre sensors.** Before 8 August the actuators were
  not all online and the baseline is contaminated; from 22 August only the centre sensors are
  real.
- **Treat consecutive blocks as correlated,** given the short washout. Dropping the first
  30–40 min of each treatment window (≈ 1 tau) before averaging removes most of the carry-over.

Excluding the 34 overtemperature episodes where the vents were shut is also worth doing, since
in those the house was thermally saturated and the actuator setting is not what determined the
climate.

### Reusable outputs

Every table above is written to `protocol check output/` as CSV by `run_all()`, so the
findings can be diffed against a re-run after the next campaign.

In [ ]:
import os
out = "protocol check output"
os.makedirs(out, exist_ok=True)
for name, df in [("blocks", rep.blocks), ("quality", rep.quality), ("flatlines", rep.flatlines),
                 ("status_codes", rep.status_codes), ("coverage", rep.coverage),
                 ("phase_a", rep.phase_a), ("dynamics", rep.dynamics), ("phase_b", rep.phase_b),
                 ("phase_c", rep.phase_c), ("phase_d", rep.phase_d), ("safety", rep.safety)]:
    df.to_csv(f"{out}/{name}.csv", index=(name == "blocks"))
print("written to", out + "/:")
print("  " + "\n  ".join(sorted(os.listdir(out))))